# Data Processing Pipeline

This notebook prepares tract-level datasets used for the location analysis model.

The pipeline integrates multiple data sources and produces cleaned tract-level feature tables.

Input datasets:
- LA Crime Dataset (2020–Present)
- American Community Survey (ACS) demographics
- TIGER Census Tracts (for land area)
- OpenStreetMap retail POIs
- Whole Foods store locations

Output tables:
- `tract_crime.csv` – crime statistics by tract
- `tract_acs.csv` – demographic features
- `tract_osm.csv` – retail density features
- `tract_wf.csv` – Whole Foods presence indicator

All datasets are aggregated to the **census tract level (GEOID)** for downstream modeling.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Crime Dataset

Load the Los Angeles crime dataset used to compute tract-level crime indicators.

The dataset contains reported crimes from the LAPD system.  
Crime incidents are aggregated by census tract to produce:

- violent crimes per 1000 residents
- property crimes per 1000 residents
- a binary indicator for low-crime tracts

In [ ]:
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/msba405-group-project/"
RAW_PATH = BASE_PATH + "raw_data/"

crime_path = RAW_PATH + "crime/la_crime_2020_present.csv"

# load dataset
df_raw = pd.read_csv(crime_path)

# preview
df_raw.head()

In [ ]:
# check column names
df_raw.columns

In [ ]:
# config
YEAR_MIN = 2022
YEAR_MAX = 2024

# rough LA bounding box
MIN_LAT = 33.2
MAX_LAT = 34.95
MIN_LON = -119.10
MAX_LON = -117.40

# crime keywords (coarse mapping)
VIOLENT_KEYWORDS = ["HOMICIDE", "ASSAULT", "ROBBERY"]
PROPERTY_KEYWORDS = ["BURGLARY", "THEFT", "CRIMINAL DAMAGE"]

In [ ]:
def report_rows(step_name, df_in):
    print(step_name, "rows =", len(df_in))

In [ ]:
# create working copy
df = df_raw.copy()

# standardize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

report_rows("after column cleanup", df)
df.columns

In [ ]:
# preview key variables
df[["date_occ", "crm_cd_desc", "lat", "lon"]].head()

In [ ]:
# convert date column
df["date_occ"] = pd.to_datetime(df["date_occ"], errors="coerce")

# convert coordinates
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# clean crime description
df["crm_cd_desc"] = df["crm_cd_desc"].astype(str).str.strip().str.upper()

# drop invalid dates and missing coordinates
df = df[df["date_occ"].notna()]
df = df[df["lat"].notna() & df["lon"].notna()]

report_rows("after type conversion", df)

In [ ]:
# check data types
df[["date_occ", "lat", "lon"]].dtypes

# check missing values
df[["date_occ", "lat", "lon"]].isna().sum()

In [ ]:
# remove zero coordinates
df = df[(df["lat"] != 0) & (df["lon"] != 0)]

report_rows("after remove zero coords", df)

In [ ]:
# rough LA bounding box
df = df[(df["lat"] >= MIN_LAT) & (df["lat"] <= MAX_LAT)]
df = df[(df["lon"] >= MIN_LON) & (df["lon"] <= MAX_LON)]

report_rows("after bbox filter", df)

In [ ]:
# coordinate summary
df[["lat", "lon"]].describe()

In [ ]:
# extract year
df["year"] = df["date_occ"].dt.year

# keep target years
df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)]

report_rows("after year filter", df)
df["year"].value_counts().sort_index()

In [ ]:
# remove duplicate cases
df["dr_no"] = df["dr_no"].astype(str).str.strip()
df = df.drop_duplicates(subset=["dr_no"])

report_rows("after dedup", df)

In [ ]:
# simple keyword-based classification (coarse)
def classify_crime(desc):
    desc = str(desc).upper()

    for k in VIOLENT_KEYWORDS:
        if k in desc:
            return "violent"

    for k in PROPERTY_KEYWORDS:
        if k in desc:
            return "property"

    return "other"

In [ ]:
# classify crimes
df["crime_type"] = df["crm_cd_desc"].apply(classify_crime)

df["crime_type"].value_counts()

In [ ]:
# keep violent and property crimes
df_rel = df[df["crime_type"] != "other"].copy()
report_rows("after keep relevant", df_rel)

# select final columns
df_rel_out = df_rel[[
    "date_occ",
    "year",
    "crime_type",
    "lat",
    "lon"
]].copy()

report_rows("final output", df_rel_out)
df_rel_out.head()

In [ ]:
# final checks
print("missing lat/lon:")
print(df_rel_out[["lat", "lon"]].isna().sum())

print("year range:", df_rel_out["year"].min(), df_rel_out["year"].max())

print("crime_type counts:")
print(df_rel_out["crime_type"].value_counts())

df_rel_out.info()
df_rel_out.describe()

In [ ]:
# save cleaned dataset
df_rel_out.to_parquet("crime_points_2022_2024_relevant.parquet", index=False)
df_rel_out.to_csv("crime_points_2022_2024_relevant.csv", index=False)

len(df_rel_out)

## Census Tract Boundaries (TIGER/Line)

Load census tract polygons from the U.S. Census Bureau TIGER/Line dataset.

These tract boundaries define the geographic units used throughout the analysis.  
They are used to:

- assign spatial data (crime points, retail POIs, store locations) to census tracts
- obtain tract land area for density calculations
- provide the GEOID identifier used to merge tract-level datasets

In [ ]:
# Install
!pip install geopandas

In [ ]:
import geopandas as gpd

tiger_path = RAW_PATH + "tiger/tl_2024_06_tract.shp"

tracts = gpd.read_file(tiger_path)

# preview
tracts.head()

In [ ]:
# check columns
tracts.columns

In [ ]:
# keep LA County tracts (COUNTYFP = 037)
la_tracts = tracts[tracts["COUNTYFP"] == "037"].copy()
la_tracts = la_tracts[["GEOID", "ALAND", "geometry"]].copy()

len(la_tracts)

In [ ]:
# check CRS
la_tracts.crs

In [ ]:
# ensure WGS84 for point coordinates
la_tracts = la_tracts.to_crs("EPSG:4326")

la_tracts.crs

In [ ]:
# create crime points GeoDataFrame
crime_gdf = gpd.GeoDataFrame(
    df_rel_out,
    geometry=gpd.points_from_xy(df_rel_out["lon"], df_rel_out["lat"]),
    crs="EPSG:4326"
)

crime_gdf.head()

In [ ]:
# assign tract GEOID to each crime point
crime_with_tract = gpd.sjoin(
    crime_gdf,
    la_tracts,
    how="inner",
    predicate="within"
)

crime_with_tract[["date_occ", "year", "crime_type", "lat", "lon", "GEOID"]].head()

In [ ]:
# tract-level crime counts
tract_crime_counts = (
    crime_with_tract
    .groupby(["GEOID", "crime_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

tract_crime_counts["total_relevant"] = (
    tract_crime_counts["property"] + tract_crime_counts["violent"]
)

tract_crime_counts.head()

In [ ]:
# add back tracts with zero crimes
tract_crime_full = (
    la_tracts[["GEOID"]]
    .merge(tract_crime_counts, on="GEOID", how="left")
    .fillna(0)
)

tract_crime_full[["property", "violent", "total_relevant"]] = (
    tract_crime_full[["property", "violent", "total_relevant"]].astype(int)
)

tract_crime_full.head()

## ACS Demographic Data

Load tract-level demographic data from the American Community Survey (ACS) for 2022–2024.

This section constructs tract-level demographic features, including:

- population (used for crime rate calculations)
- median household income
- bachelor’s degree attainment rate
- population density

In [ ]:
# load ACS population files
acs_path = RAW_PATH + "acs/"

acs22 = pd.read_csv(acs_path + "ACSDT5Y2022.B01003-Data.csv")
acs23 = pd.read_csv(acs_path + "ACSDT5Y2023.B01003-Data.csv")
acs24 = pd.read_csv(acs_path + "ACSDT5Y2024.B01003-Data.csv")

In [ ]:
# clean one ACS file
def acs_clean_one_year(df_raw, value_col, new_col_name):
    df = df_raw.iloc[1:].copy()

    # extract tract GEOID
    df["GEOID"] = df["GEO_ID"].astype(str).str[-11:]

    # convert estimate column
    df[new_col_name] = pd.to_numeric(df[value_col], errors="coerce")

    # keep required columns
    df = df[["GEOID", new_col_name]].copy()

    return df


# merge 2022-2024 ACS files
def acs_merge_3years(df22_raw, df23_raw, df24_raw, value_col, base_name):
    d22 = acs_clean_one_year(df22_raw, value_col, f"{base_name}_2022")
    d23 = acs_clean_one_year(df23_raw, value_col, f"{base_name}_2023")
    d24 = acs_clean_one_year(df24_raw, value_col, f"{base_name}_2024")

    out = d22.merge(d23, on="GEOID", how="outer")
    out = out.merge(d24, on="GEOID", how="outer")

    # keep LA County tracts
    out = out[out["GEOID"].astype(str).str.startswith("06037")].copy()

    # average across 3 years
    out[f"{base_name}_avg_22_24"] = (
        out[f"{base_name}_2022"] +
        out[f"{base_name}_2023"] +
        out[f"{base_name}_2024"]
    ) / 3

    return out

In [ ]:
# build population features
acs_pop_la = acs_merge_3years(
    acs22,
    acs23,
    acs24,
    value_col="B01003_001E",
    base_name="pop"
)

acs_pop_la.head()

In [ ]:
# merge population into tract crime counts
crime_acs = tract_crime_full.merge(
    acs_pop_la[["GEOID", "pop_avg_22_24"]],
    on="GEOID",
    how="left"
)

crime_acs.head()

In [ ]:
# crime rate per 1,000 residents
crime_acs["violent_per_1000"] = (
    crime_acs["violent"] / crime_acs["pop_avg_22_24"] * 1000
)

crime_acs["property_per_1000"] = (
    crime_acs["property"] / crime_acs["pop_avg_22_24"] * 1000
)

crime_acs[["GEOID", "violent_per_1000", "property_per_1000"]].head()

In [ ]:
# keep tracts with sufficient population
crime_acs_valid = crime_acs[crime_acs["pop_avg_22_24"] >= 500].copy()

crime_acs_valid.shape

In [ ]:
# 60th percentile thresholds
violent_cut = crime_acs_valid["violent_per_1000"].quantile(0.60)
property_cut = crime_acs_valid["property_per_1000"].quantile(0.60)

violent_cut, property_cut

In [ ]:
# flag lower-crime tracts
crime_acs_valid["crime_low_60"] = (
    (crime_acs_valid["violent_per_1000"] <= violent_cut) &
    (crime_acs_valid["property_per_1000"] <= property_cut)
)

crime_acs_valid["crime_low_60"].value_counts()

In [ ]:
# build tract-level crime feature table
tract_crime = crime_acs_valid[
    [
        "GEOID",
        "violent_per_1000",
        "property_per_1000",
        "crime_low_60"
    ]
].copy()

tract_crime.head()

In [ ]:
# load ACS income files
inc22 = pd.read_csv(acs_path + "ACSDT5Y2022.B19013-Data.csv")
inc23 = pd.read_csv(acs_path + "ACSDT5Y2023.B19013-Data.csv")
inc24 = pd.read_csv(acs_path + "ACSDT5Y2024.B19013-Data.csv")

In [ ]:
# build income features
acs_income_la = acs_merge_3years(
    inc22,
    inc23,
    inc24,
    value_col="B19013_001E",
    base_name="income"
)

acs_income_la.head()

In [ ]:
# load ACS education files
edu22 = pd.read_csv(acs_path + "ACSDT5Y2022.B15003-Data.csv")
edu23 = pd.read_csv(acs_path + "ACSDT5Y2023.B15003-Data.csv")
edu24 = pd.read_csv(acs_path + "ACSDT5Y2024.B15003-Data.csv")

In [ ]:
# build bachelor's-or-higher rate
def acs_clean_ba_rate(df_raw, new_col_name):
    df = df_raw.iloc[1:].copy()

    # extract tract GEOID
    df["GEOID"] = df["GEO_ID"].astype(str).str[-11:]

    # total population age 25+
    total_25 = pd.to_numeric(df["B15003_001E"], errors="coerce")

    # bachelor's or higher
    ba = pd.to_numeric(df["B15003_022E"], errors="coerce")
    ma = pd.to_numeric(df["B15003_023E"], errors="coerce")
    prof = pd.to_numeric(df["B15003_024E"], errors="coerce")
    phd = pd.to_numeric(df["B15003_025E"], errors="coerce")

    ba_plus = ba + ma + prof + phd

    # compute rate
    df[new_col_name] = ba_plus / total_25

    # keep required columns
    df = df[["GEOID", new_col_name]].copy()

    return df

In [ ]:
# build education features
edu_22 = acs_clean_ba_rate(edu22, "ba_rate_2022")
edu_23 = acs_clean_ba_rate(edu23, "ba_rate_2023")
edu_24 = acs_clean_ba_rate(edu24, "ba_rate_2024")

acs_edu_la = edu_22.merge(edu_23, on="GEOID", how="outer")
acs_edu_la = acs_edu_la.merge(edu_24, on="GEOID", how="outer")

# keep LA County tracts
acs_edu_la = acs_edu_la[
    acs_edu_la["GEOID"].astype(str).str.startswith("06037")
].copy()

# average across 3 years
acs_edu_la["ba_rate"] = (
    acs_edu_la[
        [
            "ba_rate_2022",
            "ba_rate_2023",
            "ba_rate_2024"
        ]
    ].mean(axis=1)
)

acs_edu_la.head()

In [ ]:
# build land area table from TIGER
aland_table = la_tracts[["GEOID", "ALAND"]].copy()

aland_table["GEOID"] = aland_table["GEOID"].astype(str)
aland_table["land_sqkm"] = aland_table["ALAND"] / 1_000_000

aland_table.head()

In [ ]:
# merge ACS features
tract_acs = acs_pop_la[["GEOID", "pop_avg_22_24"]].merge(
    acs_income_la[["GEOID", "income_avg_22_24"]],
    on="GEOID",
    how="left"
)

tract_acs = tract_acs.merge(
    acs_edu_la[["GEOID", "ba_rate"]],
    on="GEOID",
    how="left"
)

tract_acs.head()

In [ ]:
# add land area and compute population density
tract_acs["GEOID"] = tract_acs["GEOID"].astype(str)

tract_acs = tract_acs.merge(
    aland_table[["GEOID", "land_sqkm"]],
    on="GEOID",
    how="left",
    validate="one_to_one"
)

tract_acs["pop_density"] = (
    tract_acs["pop_avg_22_24"] / tract_acs["land_sqkm"]
)

tract_acs[
    ["GEOID", "land_sqkm", "pop_avg_22_24", "pop_density"]
].head()

In [ ]:
# build final ACS feature table
tract_acs["income"] = tract_acs["income_avg_22_24"]
tract_acs["pop"] = tract_acs["pop_avg_22_24"]

tract_acs = tract_acs[
    [
        "GEOID",
        "income",
        "ba_rate",
        "pop",
        "pop_density"
    ]
].copy()

tract_acs.head()

In [ ]:
# preview tract_crime
tract_crime.head()

In [ ]:
# preview tract_acs
tract_acs.head()

In [ ]:
# check table shapes
tract_crime.shape, tract_acs.shape

## OSM Retail and Amenity Data

Load OpenStreetMap point data and keep selected retail and service locations.

OSM points are spatially assigned to census tracts.  
This section constructs a tract-level retail density feature based on the number of retail and service locations per square kilometer.

In [ ]:
# load OSM point shapefile
osm_path = RAW_PATH + "osm/points.shp"

points = gpd.read_file(osm_path)

points.head()

In [ ]:
# check columns
points.columns

In [ ]:
# selected retail and service types
retail_types = [
    "restaurant",
    "fast_food",
    "cafe",
    "bar",
    "pharmacy",
    "bank",
    "atm",
    "fuel",
    "ice_cream",
    "dentist",
    "clinic",
    "post_office",
    "post_box",
    "bicycle_rental",
    "vending_machine"
]

In [ ]:
# keep selected OSM points
retail_points = points[points["type"].isin(retail_types)].copy()

retail_points["type"].value_counts()

In [ ]:
# check point count
retail_points.shape

In [ ]:
# match CRS to tract polygons
retail_points = retail_points.to_crs(la_tracts.crs)

In [ ]:
# assign tract GEOID to each OSM point
points_tract = gpd.sjoin(
    retail_points,
    la_tracts[["GEOID", "geometry"]],
    how="inner",
    predicate="within"
)

points_tract.head()

In [ ]:
# count OSM points by tract
retail_counts = (
    points_tract
    .groupby("GEOID")
    .size()
    .reset_index(name="retail_count")
)

retail_counts.head()

In [ ]:
# build land area table
aland_table = la_tracts[["GEOID", "ALAND"]].copy()

aland_table["GEOID"] = aland_table["GEOID"].astype(str)
aland_table["land_sqkm"] = aland_table["ALAND"] / 1_000_000

aland_table.head()

In [ ]:
# add land area and compute retail density
retail_counts["GEOID"] = retail_counts["GEOID"].astype(str)

retail_counts = retail_counts.merge(
    aland_table[["GEOID", "land_sqkm"]],
    on="GEOID",
    how="left",
    validate="one_to_one"
)

retail_counts["retail_density"] = (
    retail_counts["retail_count"] / retail_counts["land_sqkm"]
)

retail_counts.head()

In [ ]:
# add back tracts with zero retail points
tract_osm = (
    aland_table[["GEOID", "land_sqkm"]]
    .merge(retail_counts[["GEOID", "retail_count", "retail_density"]],
           on="GEOID",
           how="left")
    .fillna(0)
)

tract_osm = tract_osm[["GEOID", "retail_density"]].copy()

tract_osm.head()

In [ ]:
# check final OSM feature table
tract_osm.head()
tract_osm.shape

## Whole Foods Store Locations

Load Whole Foods store locations in Los Angeles and convert them to spatial points.

Store locations are spatially joined to census tracts.  
This section constructs a tract-level binary indicator showing whether a tract contains a Whole Foods store.

In [ ]:
# load Whole Foods locations
wf_path = RAW_PATH + "wf/wf_locations_la.xlsx"

wf_df = pd.read_excel(wf_path)

wf_df.head()

In [ ]:
# standardize column names
wf_df = wf_df.rename(columns={
    "Location": "location",
    "Latitude": "lat",
    "Longitude": "lon"
})

wf_df.head()

In [ ]:
# create store points GeoDataFrame
wf_gdf = gpd.GeoDataFrame(
    wf_df.copy(),
    geometry=gpd.points_from_xy(wf_df["lon"], wf_df["lat"]),
    crs="EPSG:4326"
)

wf_gdf.head()

In [ ]:
# assign tract GEOID to each store
wf_join = gpd.sjoin(
    wf_gdf,
    la_tracts[["GEOID", "geometry"]],
    how="inner",
    predicate="within"
)

wf_join[["location", "lat", "lon", "GEOID"]].head()

In [ ]:
# keep tracts with Whole Foods
wf_tracts = wf_join[["GEOID"]].drop_duplicates().copy()

wf_tracts["has_wf"] = 1

wf_tracts.head()

In [ ]:
# add back all tracts
all_tracts = la_tracts[["GEOID"]].drop_duplicates().copy()

tract_wf = all_tracts.merge(
    wf_tracts,
    on="GEOID",
    how="left"
)

tract_wf["has_wf"] = tract_wf["has_wf"].fillna(0).astype(int)

tract_wf.head()

In [ ]:
# check final WF feature table
tract_wf["has_wf"].value_counts()

In [ ]:
# check table shape
tract_wf.shape

## Export Final Feature Tables

Export tract-level feature tables for downstream analysis and modeling.

The exported datasets include:

- tract_crime: crime rates and crime indicator
- tract_acs: demographic and socioeconomic features
- tract_osm: retail and amenity density
- tract_wf: Whole Foods store presence

In [ ]:
PROC_PATH = BASE_PATH + "processed_data/"

In [ ]:
# export final feature tables
tract_crime.to_csv(PROC_PATH + "tract_crime.csv", index=False)
tract_acs.to_csv(PROC_PATH + "tract_acs.csv", index=False)
tract_osm.to_csv(PROC_PATH + "tract_osm.csv", index=False)
tract_wf.to_csv(PROC_PATH + "tract_wf.csv", index=False)

print("Export completed.")

In [ ]:
# check exported tables
print("tract_crime:", tract_crime.shape)
print("tract_acs:", tract_acs.shape)
print("tract_osm:", tract_osm.shape)
print("tract_wf:", tract_wf.shape)